# Analisis Angka Harapan Hidup

Dua kumpulan data yang mengeksplorasi angka harapan hidup global:
- **Gapminder** (1952–2007): country, year, population, continent, lifeExp, gdpPercap
- **Angka Harapan Hidup WHO** (2000–2015): 193 negara, 22 indikator (mortalitas, BMI, PDB, pendidikan, dll.)

Buku kerja ini mendemonstrasikan impor file CSV dan analisis data dalam **Python** dan **R**.

## 1. Pengaturan: Pasang paket & unduh kumpulan data

In [ ]:
import micropip
await micropip.install(['pandas', 'plotly'])
print('Terpasang pandas + plotly')

import pyodide.http, os

datasets = {
    "gapminder.csv": "https://raw.githubusercontent.com/resbaz/r-novice-gapminder-files/master/data/gapminder-FiveYearData.csv",
    "who_life_expectancy.csv": "https://raw.githubusercontent.com/Sid-149/Life-Expectancy-Predictor-Comparative-Analysis/main/Notebooks/Life%20Expectancy%20Data.csv"
}

os.makedirs("/shared/data", exist_ok=True)

for name, url in datasets.items():
    path = f"/shared/data/{name}"
    if os.path.exists(path):
        print(f"Sudah ada: {path}")
    else:
        resp = await pyodide.http.pyfetch(url)
        text = await resp.string()
        with open(path, "w") as f:
            f.write(text)
        lines = text.count("\n")
        print(f"Diunduh {name}: {lines} baris")

## 2. Gapminder: Eksplorasi dengan Python

In [ ]:
import pandas as pd

gap = pd.read_csv("/shared/data/gapminder.csv")
print(f"Dimensi: {gap.shape}")
print(f"Benua: {sorted(gap['continent'].unique())}")
print(f"Rentang tahun: {gap['year'].min()}-{gap['year'].max()}")
print()
gap.describe()

In [ ]:
import plotly.express as px
import json, js
from plotly.utils import PlotlyJSONEncoder

def plain_plotly(value):
    if hasattr(value, 'tolist'):
        return value.tolist()
    if isinstance(value, dict):
        return {key: plain_plotly(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [plain_plotly(item) for item in value]
    return value

def show_plotly(fig):
    payload = plain_plotly(fig.to_plotly_json())
    js.renderPlot(json.dumps({"traces": payload["data"], "layout": payload["layout"]}, cls=PlotlyJSONEncoder))

# Angka harapan hidup dari waktu ke waktu berdasarkan benua
avg = gap.groupby(['year', 'continent'])['lifeExp'].mean().reset_index()
fig = px.line(avg, x='year', y='lifeExp', color='continent',
              title='Angka Harapan Hidup Berdasarkan Benua (1952-2007)',
              labels={'lifeExp': 'Angka Harapan Hidup (tahun)', 'year': 'Tahun'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

In [ ]:
# PDB vs Angka Harapan Hidup (2007), ukuran gelembung = populasi
g2007 = gap[gap['year'] == 2007]
fig = px.scatter(g2007, x='gdpPercap', y='lifeExp', size='pop',
                 color='continent', hover_name='country',
                 log_x=True, size_max=50,
                 title='PDB vs Angka Harapan Hidup (2007)',
                 labels={'gdpPercap': 'PDB per kapita (log)', 'lifeExp': 'Angka Harapan Hidup'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 3. Gapminder: Eksplorasi dengan R

In [ ]:
gap <- read.csv("/shared/data/gapminder.csv")
str(gap)
summary(gap$lifeExp)

In [ ]:
# Distribusi angka harapan hidup berdasarkan benua (boxplot)
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
boxplot(lifeExp ~ continent, data = gap,
        main = "Angka Harapan Hidup Berdasarkan Benua",
        xlab = "Benua", ylab = "Angka Harapan Hidup (tahun)",
        col = c("#636EFA", "#EF553B", "#00CC96", "#AB63FA", "#FFA15A"),
        border = "white")

In [ ]:
# 10 negara teratas berdasarkan peningkatan angka harapan hidup (1952 vs 2007)
early <- gap[gap$year == 1952, c("country", "lifeExp")]
late  <- gap[gap$year == 2007, c("country", "lifeExp")]
merged <- merge(early, late, by = "country", suffixes = c("_1952", "_2007"))
merged$improvement <- merged$lifeExp_2007 - merged$lifeExp_1952
top10 <- head(merged[order(-merged$improvement), ], 10)

par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white", mar = c(5, 10, 4, 2))
barplot(top10$improvement, names.arg = top10$country,
        horiz = TRUE, las = 1,
        main = "10 Teratas: Peningkatan Angka Harapan Hidup (1952-2007)",
        xlab = "Tahun yang diperoleh",
        col = "#00CC96", border = NA)

## 4. Angka Harapan Hidup WHO: Eksplorasi dengan Python

In [ ]:
who = pd.read_csv("/shared/data/who_life_expectancy.csv")
print(f"Dimensi: {who.shape}")
print(f"Kolom: {list(who.columns)}")
print(f"\nNilai yang hilang (5 teratas):")
print(who.isnull().sum().sort_values(ascending=False).head())
print()
who.head()

In [ ]:
# Negara Berkembang vs Maju: distribusi angka harapan hidup yang telah dikelompokkan sebelumnya
# Koordinat batang eksplisit dirender secara konsisten melalui jembatan Plotly peramban.
import numpy as np
life = who.dropna(subset=['Life expectancy'])
edges = np.linspace(life['Life expectancy'].min(), life['Life expectancy'].max(), 41)
bin_width = edges[1] - edges[0]
hist_rows = []
for status, group in life.groupby('Status'):
    counts, _ = np.histogram(group['Life expectancy'], bins=edges)
    hist_rows.extend({
        'Life expectancy': float(left + bin_width / 2),
        'Count': int(count),
        'Status': status
    } for left, count in zip(edges[:-1], counts))

hist = pd.DataFrame(hist_rows)
fig = px.bar(hist, x='Life expectancy', y='Count', color='Status',
             barmode='overlay', opacity=0.7,
             title='Angka Harapan Hidup: Berkembang vs Maju',
             labels={'Life expectancy': 'Angka Harapan Hidup (tahun)'})
fig.update_traces(width=float(bin_width * 0.92))
fig.update_layout(template='plotly_dark', bargap=0.03)
show_plotly(fig)

In [ ]:
# Pendidikan vs Angka Harapan Hidup
w2014 = who[who['Year'] == 2014].dropna(subset=['Schooling', 'Life expectancy'])
fig = px.scatter(w2014, x='Schooling', y='Life expectancy',
                 color='Status', hover_name='Country',
                 title='Pendidikan vs Angka Harapan Hidup (2014)',
                 labels={'Life expectancy': 'Angka Harapan Hidup (tahun)',
                         'Schooling': 'Lama Sekolah (tahun)'})
fig.update_layout(template='plotly_dark')
show_plotly(fig)

## 5. Angka Harapan Hidup WHO: Eksplorasi dengan R

In [ ]:
who <- read.csv("/shared/data/who_life_expectancy.csv")
str(who)
cat("\nNegara:", length(unique(who$Country)))
cat("\nRentang tahun:", range(who$Year))

In [ ]:
# Korelasi: Mortalitas Dewasa vs Angka Harapan Hidup
par(bg = "#1e1e1e", fg = "white", col.axis = "white",
    col.lab = "white", col.main = "white")
plot(who$Adult.Mortality, who$Life.expectancy,
     pch = 16, cex = 0.5,
     col = ifelse(who$Status == "Developed", "#636EFA80", "#EF553B80"),
     main = "Mortalitas Dewasa vs Angka Harapan Hidup",
     xlab = "Mortalitas Dewasa (per 1000)",
     ylab = "Angka Harapan Hidup (tahun)")
legend("topright", legend = c("Maju", "Berkembang"),
       col = c("#636EFA", "#EF553B"), pch = 16, text.col = "white")

In [ ]:
# Model linier sederhana: apa yang memprediksi angka harapan hidup?
who_clean <- na.omit(who[, c("Life.expectancy", "Schooling",
                              "Adult.Mortality", "GDP", "BMI")])
model <- lm(Life.expectancy ~ Schooling + Adult.Mortality + log1p(GDP) + BMI,
            data = who_clean)
summary(model)

## Kesimpulan Utama

- Angka harapan hidup meningkat secara global, namun kesenjangan besar antarbenua tetap ada
- PDB dan pendidikan merupakan prediktor positif yang kuat untuk angka harapan hidup
- Kematian dewasa adalah prediktor negatif terkuat
- Negara-negara berkembang menunjukkan varians yang jauh lebih besar dalam hasilnya